# CV Masterclass 15: Unsupervised Anomaly Detection

Welcome to Notebook 18. In previous modules, we mastered supervised learning, where we have thousands of examples per class. But in the real world—specifically industrial quality control—you might have 1,000,000 "normal" images and only 5 examples of a "scratched surface."

Anomaly Detection (AD) is the art of learning **only from normal data** and identifying anything that deviates from that norm. Today, we will explore the state-of-the-art algorithms: **PatchCore** and **PaDiM**.

---

## 🎓 Socratic Deep Check
Ponder these questions before we begin:

> *"If a model sees 10,000 images of perfect apples and then sees a banana, is it an anomaly or just a new class? What defines 'anomalous' vs. 'out-of-distribution'?"*

> *"Why does a standard classification network (like ResNet) fail when you only give it one class? Where exactly does the math break down during the Softmax evaluation?"*

---

1. **The Curse of Imbalance:** Why Supervised Learning Fails.
2. **PatchCore:** Local Feature Memory & Coresets.
3. **Feature Extraction:** Leveraging Modern Pre-trained Backbones.
4. **Student-Teacher Distillation:** Detecting via Inconsistency.
5. **Evaluation:** AUROC (Area Under ROC Curve) from Scratch.
6. **PaDiM:** Patch Distribution Modeling & Mahalanobis Distance.
7. **Industrial Deployment:** Scaling to Production Specs.

## 1. The Curse of Imbalance

In supervised learning, we minimize Cross-Entropy between predicted and ground-truth labels. This works because the gradient signal pushes the decision boundary *between* classes.

In **Anomaly Detection**, there is no "other" class boundary. If you train a ResNet to classify "Normal", it will simply learn that every image has probability 1.0 of being normal. It never learns what *isn't* normal.

### The Unsupervised Solution
Instead of labels, we learn the **Distribution of Normality** $P(x_{normal})$. We flags $x$ as anomalous if $P(x) < \tau$.
- **Feature-based:** Extract embeddings from pre-trained networks and compare distances (PatchCore).
- **Probabilistic:** Fit a Gaussian to patch-level feature distributions (PaDiM).
- **Reconstruction-based:** Train an Autoencoder to reconstruct normal images; anomalies are marked by high reconstruction error.

## 2. PatchCore: Local Feature Memory

PatchCore (2022) is currently one of the highest-performing algorithms for industrial anomaly detection. It works by creating a "Memory Bank" of local patch features.

1.  **Extract:** Run normal images through a pre-trained CNN (WideResNet).
2.  **Patchify:** Instead of global features, we take spatial feature maps (H' x W') and treat each pixel (a patch embedding) as a memory entry.
3.  **Coreset Selection:** The memory bank would be too massive to search at test time. We use **Greedy k-center selection** to pick a tiny fraction (~1%) of the features that still cover the original distribution.
4.  **Score:** At test time, an anomaly score is the distance to the nearest neighbor in the coreset memory bank.

## 📑 Table of Contents

* [🎓 Socratic Deep Check](#socratic-deep-check)
* [1. The Curse of Imbalance](#1-the-curse-of-imbalance)
  * [The Unsupervised Solution](#the-unsupervised-solution)
* [2. PatchCore: Local Feature Memory](#2-patchcore-local-feature-memory)
* [4. Student-Teacher Distillation](#4-student-teacher-distillation)
* [5. Evaluation: AUROC (Area Under ROC Curve) from Scratch](#5-evaluation-auroc-area-under-roc-curve-from-scratch)
* [6. PaDiM: Patch Distribution Modeling](#6-padim-patch-distribution-modeling)
  * [WHY Mahalanobis over Euclidean?](#why-mahalanobis-over-euclidean)
* [7. Deep Socratic Synthesis](#7-deep-socratic-synthesis)


### 🎓 Junior to Senior: Concept Bridge

**Junior:** Tries to train a classic binary classification model ('Normal' vs 'Anomaly'). Inevitably fails entirely when an unseen type of broken widget appears.

**Senior:** Models Anomaly Detection fundamentally purely as an Unsupervised Out-Of-Distribution (OOD) formulation. Models the distribution of only 'Normal' data using Autoencoders or Normalizing Flows, flagging anything with high structural reconstruction error as anomalies.

---


### 🎓 Junior to Senior: Concept Bridge

**Junior:** Tries to train a classic binary classification model ('Normal' vs 'Anomaly'). Inevitably fails entirely when an unseen type of broken widget appears.

**Senior:** Models Anomaly Detection fundamentally purely as an Unsupervised Out-Of-Distribution (OOD) formulation. Models the distribution of only 'Normal' data using Autoencoders or Normalizing Flows, flagging anything with high structural reconstruction error as anomalies.

---


### 🎓 Junior to Senior: Concept Bridge

**Junior:** Tries to train a classic binary classification model ('Normal' vs 'Anomaly'). Inevitably fails entirely when an unseen type of broken widget appears.

**Senior:** Models Anomaly Detection fundamentally purely as an Unsupervised Out-Of-Distribution (OOD) formulation. Models the distribution of only 'Normal' data using Autoencoders or Normalizing Flows, flagging anything with high structural reconstruction error as anomalies.

---


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class PatchCoreExtractor(nn.Module):
    """
    Extracts mid-level features for PatchCore anomaly detection.
    """
    def __init__(self):
        super().__init__()
        # FIX 1: Modern torchvision weights API and correct function name
        try:
            from torchvision.models import Wide_ResNet50_2_Weights
            self.backbone = models.wide_resnet50_2(weights=Wide_ResNet50_2_Weights.DEFAULT)
            print("Using modern torchvision Wide ResNet 50-2 ✓")
        except (ImportError, AttributeError):
            # Fallback for older versions
            self.backbone = models.wide_resnet50_2(pretrained=True)
            print("Warning: Using deprecated pretrained=True")
            
        self.backbone.eval()
        for param in self.backbone.parameters():
            param.requires_grad = False

    def forward(self, x):
        # Extract intermediate layers (layer2 and layer3 provide best detail/semantic balance)
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)

        x1 = self.backbone.layer1(x)
        x2 = self.backbone.layer2(x1)
        x3 = self.backbone.layer3(x2)
        return x2, x3

# TEST IT
extractor = PatchCoreExtractor()
dummy_in = torch.randn(1, 3, 224, 224)
f2, f3 = extractor(dummy_in)
print(f"Layer 2 Features: {f2.shape}") # Expected: [1, 512, 28, 28]
print(f"Layer 3 Features: {f3.shape}") # Expected: [1, 1024, 14, 14]
assert f2.shape[1] == 512 and f3.shape[1] == 1024
print("Feature extraction architecture verified ✓")

## 4. Student-Teacher Distillation

Another popular approach is **Knowledge Distillation**. A large, frozen pre-trained "Teacher" network extracts features. A smaller "Student" network is trained to mimic those features **only on normal data**.

Because the Student never saw anomalous data during training, it will fail to mimic the Teacher's features when an anomaly appears. The "mimicry error" becomes the anomaly map!

## 5. Evaluation: AUROC (Area Under ROC Curve) from Scratch

AUROC is the universal metric for anomaly detection. It measures how well the anomaly score separates normal from anomalous samples across **ALL** possible thresholds simultaneously.

**AUROC = probability that a randomly chosen anomalous sample scores higher than a randomly chosen normal sample.**

- **AUROC = 1.0:** Perfect separation (all anomalies score higher than all normals)
- **AUROC = 0.5:** Random guessing
- **AUROC = 0.0:** Inverted prediction (anomalies all score lower)

In [ ]:
import numpy as np

def compute_auroc(normal_scores, anomaly_scores, num_thresholds=1000):
    """
    Computes AUROC from scratch using the trapezoidal rule.
    normal_scores: anomaly scores of normal samples (lower is better)
    anomaly_scores: anomaly scores of anomalous samples (higher is better)
    """
    all_scores = np.concatenate([normal_scores, anomaly_scores])
    all_labels = np.concatenate([np.zeros(len(normal_scores)),
                                  np.ones(len(anomaly_scores))])
    
    thresholds = np.linspace(all_scores.min(), all_scores.max(), num_thresholds)
    
    fprs = []
    tprs = []
    
    for thresh in thresholds:
        # Predict as anomaly if score >= threshold
        pred_pos = (all_scores >= thresh)
        
        tp = ((pred_pos == 1) & (all_labels == 1)).sum()
        fp = ((pred_pos == 1) & (all_labels == 0)).sum()
        tn = ((pred_pos == 0) & (all_labels == 0)).sum()
        fn = ((pred_pos == 0) & (all_labels == 1)).sum()
        
        tpr = tp / (tp + fn + 1e-10)  # Sensitivity (True Positive Rate)
        fpr = fp / (fp + tn + 1e-10)  # Fall-out (False Positive Rate)
        
        tprs.append(tpr)
        fprs.append(fpr)
    
    # Sort by FPR for trapezoidal integration
    pairs = sorted(zip(fprs, tprs))
    fprs_sorted, tprs_sorted = zip(*pairs)
    
    # Trapezoidal rule: area under the ROC curve
    auroc = np.trapz(tprs_sorted, fprs_sorted)
    return auroc, np.array(fprs_sorted), np.array(tprs_sorted)

# TEST IT
np.random.seed(42)
# Well-separated distributions (should give high AUROC)
normal_scores_good = np.random.normal(0.2, 0.1, 1000)
anomaly_scores_good = np.random.normal(0.8, 0.1, 200)
auroc_good, fpr_good, tpr_good = compute_auroc(normal_scores_good, anomaly_scores_good)

# Overlapping distributions (should give lower AUROC)
normal_scores_bad = np.random.normal(0.5, 0.2, 1000)
anomaly_scores_bad = np.random.normal(0.6, 0.2, 200)
auroc_bad, _, _ = compute_auroc(normal_scores_bad, anomaly_scores_bad)

print(f"Well-separated AUROC: {auroc_good:.4f} (expect > 0.95)")
print(f"Overlapping AUROC:    {auroc_bad:.4f} (expect 0.5-0.7)")

assert auroc_good > 0.95, "Good separator should achieve AUROC > 0.95"
assert auroc_bad < auroc_good, "Better separation must yield higher AUROC"
print("AUROC implementation verified ✓")

## 6. PaDiM: Patch Distribution Modeling

PatchCore uses nearest-neighbor distance as the anomaly score. **PaDiM** (Patch Distribution Modeling) takes a different approach: for each spatial position (i, j) in the feature map, fit a **multivariate Gaussian** to the patch embeddings from all normal training images.

At test time, the **Mahalanobis distance** from a test patch to the fitted Gaussian at its position gives the anomaly score:
$$ D(f, \mu, \Sigma) = \sqrt{(f - \mu)^T \Sigma^{-1} (f - \mu)} $$

### WHY Mahalanobis over Euclidean?
Mahalanobis distance accounts for **correlation between feature dimensions**. Two features that are always correlated in normal images (e.g., horizontal and vertical edges at the same location) will not trigger a high score just because one is slightly elevated — only if they deviate in an **UNEXPECTED** direction does the score spike.

In [ ]:
def fit_padim_gaussians(normal_features):
    """
    Fits per-position multivariate Gaussians to normal training features.
    normal_features: (N, C, H, W) — N normal images
    Returns: dict of {(i,j): (mean, cov_inv)} for each spatial position
    """
    N, C, H, W = normal_features.shape
    gaussians = {}
    
    for i in range(H):
        for j in range(W):
            patches = normal_features[:, :, i, j]  # (N, C)
            mu = patches.mean(dim=0)
            
            # Sample covariance (regularized for invertibility)
            diff = patches - mu
            cov = (diff.T @ diff) / (N - 1)
            cov += 0.01 * torch.eye(C)  # Regularization
            
            cov_inv = torch.linalg.inv(cov)
            gaussians[(i, j)] = (mu, cov_inv)
    
    return gaussians

def mahalanobis_score(test_feature, mu, cov_inv):
    """Computes Mahalanobis distance from test feature to Gaussian."""
    diff = test_feature - mu
    return torch.sqrt(diff @ cov_inv @ diff)

# TEST IT (on tiny synthetic data for speed)
C, H, W = 16, 4, 4
N_normal = 50

# Simulate normal feature maps (correlated features)
normal_feats = torch.randn(N_normal, C, H, W)
# Make channel 0 and 1 correlated (normal pattern)
normal_feats[:, 1, :, :] = normal_feats[:, 0, :, :] + 0.1 * torch.randn(N_normal, H, W)

gaussians = fit_padim_gaussians(normal_feats)

# Score a normal test image
test_normal = torch.randn(C, H, W)
test_normal[1] = test_normal[0] + 0.1 * torch.randn(H, W)  # Same correlation

# Score an anomalous test image (correlation broken)
test_anomaly = torch.randn(C, H, W)
test_anomaly[1] = -test_anomaly[0] + torch.randn(H, W) * 2  # Anti-correlated!

score_normal = mahalanobis_score(test_normal[:, 0, 0], *gaussians[(0, 0)])
score_anomaly = mahalanobis_score(test_anomaly[:, 0, 0], *gaussians[(0, 0)])

print(f"Normal Mahalanobis score: {score_normal.item():.4f}")
print(f"Anomaly Mahalanobis score: {score_anomaly.item():.4f}")
assert score_anomaly > score_normal, "Anomaly must score higher than normal!"
print("PaDiM Mahalanobis scoring verified ✓")

---
## ✅ Knowledge Check

### Q1: Why is supervised binary classification generally inappropriate for industrial anomaly detection?
<details><summary>👉 Answer</summary>

Industrial anomalies are exceptionally rare and highly variable. You can never gather enough explicit examples of every possible 'broken' state, so the model must instead strictly learn the singular distribution mapping of 'Correct'.
</details>

### Q2: How does an Autoencoder flag anomalies without any classification heads?
<details><summary>👉 Answer</summary>

It is trained exclusively to reconstruct 'Normal' images through a compressed bottleneck. When fed an anomalous image, the bottleneck inevitably drops the unknown anomalous spatial components, resulting in a quantifiable 'pixel reconstruction delta error map'.
</details>

---
## 🔨 Practical Practice

### Exercise 1
Reconstruction Evaluator: Write python logic defining how to generate an absolute spatial heatmap representing the delta magnitude between an Autoencoder input and output.

### Exercise 2
Threshold Calibration: Write an evaluation loop establishing validation reconstruction threshold standard deviations to automatically bound maximum allowable industrial errors.


## 7. Deep Socratic Synthesis

**Q:** *Why does PatchCore benefit from using mid-level backbone features (layers 2 and 3) rather than the final global feature map?*

**A:** Because anomalies in industrial parts (scratches, holes, burns) are local spatial events. Final global features capture "Image-level semantics" (Is this a shoe or a bottle?) but destroy spatial coordinate detail. Layers 2 and 3 retain enough spatial resolution to localise the anomaly precisely.

**Q:** *What is the fundamental drawback of PaDiM compared to PatchCore?*

**A:** **The Alignment Assumption.** PaDiM assumes that every image is perfectly aligned (e.g., a manufactured part always appears in the exact same orientation and scale). If the camera moves slightly, the fitted Gaussian at coordinate (i, j) will no longer represent the same physical part of the object. PatchCore is more robust to slight misalignments because it searches globally in the memory bank.